# M3.9 · 模型监控与衰减

> 模型上线后会因市场结构变化、风格切换而衰减。监控性能、分布漂移并及时预警，是量化运维的核心。
> 本节实现滚动 IC、PSI 分布漂移、特征漂移与衰减预警。

## 一、监控的三个维度

| 维度 | 指标 | 含义 |
|---|---|---|
| 性能 | 滚动 IC/收益 | 预测力是否还在 |
| 分布漂移 | PSI | 输入/预测分布是否变化 |
| 特征漂移 | 均值/方差移位 | 单特征是否异常 |

**PSI 经验阈值：** <0.1 稳定；0.1~0.25 轻微漂移；>0.25 显著漂移。
**衰减预警：** 连续多期 IC 低于阈值即报警，触发重训或下线。


## 二、准备数据并训练基线

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
from factor_lab2 import hfactors
from factor_lab2.combine import cross_zscore
from model_lab2 import labels, monitor
import lightgbm as lgb
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
raw = D.features(codes, ['$open','$close','$high','$low','$volume'],
                 start_time='2018-01-01', freq='day')
op=raw['$open'].unstack('instrument'); cl=raw['$close'].unstack('instrument')
hi=raw['$high'].unstack('instrument'); lo=raw['$low'].unstack('instrument')
vo=raw['$volume'].unstack('instrument')
lab = labels.cs_rank_label(cl, 5)
hf = hfactors.build_hf_factors(op, cl, hi, lo, vo)
feat_names = ['已实现波动20','日内动量20','尾盘位置20','量价相关20','收益偏度60']
z = pd.concat({n: cross_zscore(hf[n]).stack() for n in feat_names}, axis=1)
z.index.names = ['datetime','instrument']
df = z.join(lab.stack().rename('y')).dropna()
cut = df.index.get_level_values('datetime').unique()
split = cut[int(len(cut)*0.8)]
tr = df[df.index.get_level_values('datetime') <= split]
te = df[df.index.get_level_values('datetime') > split]
m = lgb.LGBMRegressor(n_estimators=120, num_leaves=31, learning_rate=0.05,
                      random_state=42, verbose=-1).fit(tr[feat_names], tr['y'])
pdf = te.copy(); pdf['pred'] = m.predict(te[feat_names])
print('训练/测试:', tr.shape, te.shape)

## 三、滚动性能监控

In [ ]:
ic_ser, ic_roll = monitor.rolling_ic_by_date(pdf, window=60)
print('测试期 IC=%.4f' % ic_ser.mean())
alert = monitor.decay_alert(ic_ser.values, threshold=0.0, n_below=3)
print('衰减预警次数:', alert['n_alert'], ' 最新IC=%.4f' % alert['latest_ic'])

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(11,4.4))
ax.plot(ic_ser.index, ic_ser.rolling(20).mean(), color='#9ca3af', label='IC(20日平滑)')
ax.plot(ic_roll.index, ic_roll, color='#4f46e5', label='IC(60日平滑)')
ax.axhline(0, color='#f87171', lw=0.8)
ax.set_title('滚动 IC 性能监控'); ax.legend()
plt.tight_layout(); plt.show()

## 四、PSI 与特征漂移

In [ ]:
psis = {}
for n in feat_names:
    psis[n] = monitor.psi(tr[n].values, te[n].values)
psi_s = pd.Series(psis).sort_values(ascending=False)
print('各特征 PSI:')
print(psi_s.round(4).to_string())
drift = monitor.feature_drift(tr[feat_names], te[feat_names])
print('\n特征漂移:')
print(drift.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4.4))
colors = ['#f87171' if v>0.25 else ('#fbbf24' if v>0.1 else '#34d399')
          for v in psi_s.values]
axes[0].barh(psi_s.index[::-1], psi_s.values[::-1], color=colors[::-1])
axes[0].axvline(0.1, color='#333', ls=':'); axes[0].axvline(0.25, color='#333', ls=':')
axes[0].set_title('各特征 PSI（阈值0.1/0.25）')
x = np.arange(len(feat_names)); w=0.35
axes[1].bar(x-w/2, drift['mean_shift'], w, label='均值移位', color='#60a5fa')
axes[1].set_xticks(x); axes[1].set_xticklabels(feat_names, rotation=45, ha='right')
axes[1].axhline(0, color='#333', lw=0.8); axes[1].set_title('特征均值漂移')
axes[1].legend()
plt.tight_layout(); plt.show()

## 五、监控响应流程

1. 滚动 IC 连续低于阈值 → 报警；
2. 检查 PSI/特征漂移定位原因（风格切换、数据异常、宏观环境变化）；
3. 数据问题先修数据；模型衰减则重训，必要时回退到上一稳定版本；
4. 重训后仍不达标则下线，避免持续亏损。

## 六、小结与面试要点

**面试高频：**
- 模型为什么会衰减？（答：市场非平稳、风格切换、因子被套利、结构变化）
- PSI 原理与阈值？（答：两期分布占比的信息差异，<0.1稳定、>0.25显著漂移）
- 怎么监控模型？（答：滚动性能+分布漂移+特征漂移，分级预警）
- 报警后怎么处理？（答：定位是数据还是模型，修数据/重训/回退/下线）
- 性能监控用什么指标？（答：滚动IC、样本外收益、胜率等，区分统计噪声与真衰减）

## 动手练习
1. 计算滚动 IC 并做衰减预警；
2. 计算各特征 PSI 并标注漂移等级；
3. 描述从报警到处置的完整流程。

---
M3 模型进阶全部完成，下一批 B6 进入组合风险（上）。